# 📚 Taller Práctico — Aprendizaje Automático
**Universidad Internacional del Ecuador (UIDE) — Maestría**  
**Fecha:** 2026-08-01  

---

## Contexto

El presente informe técnico está dirigido a **María Fernanda** y la **Universidad Andina del Pacífico**.  
Se evalúa si los datos institucionales pueden sustentar un **modelo de alerta temprana** de riesgo académico.

Se trabaja sobre un dataset simulado de 100 registros de estudiantes con variables académicas y de comportamiento digital.

---

## 📦 Librerías e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual (mismo estilo que 01_clase.ipynb)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("✅ Librerías cargadas correctamente")

## 📂 Carga del Dataset

Dataset simulado generado en `dataset.ipynb` con semilla `np.random.seed(42)` para reproducibilidad.  
Contiene **100 registros** de estudiantes con información académica y de conectividad.

In [ ]:
# ── Reproducción del dataset (mismo código que dataset.ipynb) ──────────────
np.random.seed(42)
n = 100

data = {
    "id_estudiante": [f"EST-{i:03d}" for i in range(1, 95)] + ["EST-010", "EST-012", "EST-025", "EST-030", "EST-050", "EST-001"],
    "sede": np.random.choice(["Quito", "Guayaquil", "Cuenca", "quito_2"], n),
    "modalidad": np.random.choice(["Presencial", "En línea"], n),
    "promedio_acumulado": np.random.uniform(5.0, 10.0, n),
    "tiempo_conexion_min": np.random.exponential(scale=120, size=n),
    "entregas_semana_1_8": np.random.randint(0, 10, n),
    "nivel_satisfaccion": np.random.choice([1, 2, 3, 4, 5, np.nan], n),
    "calificacion_final_semestre": np.random.uniform(0, 10, n),   # ⚠️ variable futura
    "fecha_ultimo_acceso": pd.date_range(start="2026-01-15", periods=n, freq="D").strftime("%Y/%m/%d")
}

df_raw = pd.DataFrame(data)
print(f"Dimensiones iniciales del dataset: {df_raw.shape}")
df_raw.head(10)

---

# 🔷 ÍTEM 1: Definición de Unidad de Análisis e Ingesta

**Enunciado:**  
> Defina la unidad de análisis (estudiante-semestre o estudiante-asignatura) y ejecute en Pandas el código de agregación necesario para estructurar la tabla final.

---

## 1.1 Justificación de la Unidad de Análisis

La **unidad de análisis seleccionada es `estudiante-semestre`**, dado que:

| Criterio | Estudiante-Semestre ✅ | Estudiante-Asignatura ❌ |
|---|---|---|
| Granularidad | Nivel de seguimiento semestral | Requiere datos por materia (no disponibles) |
| Variables disponibles | `promedio_acumulado`, `entregas_semana_1_8`, `tiempo_conexion_min` son métricas semestral-globales | Estas variables no están desagregadas por asignatura |
| Objetivo del modelo | Alerta temprana de riesgo por **período académico** | No hay columna `asignatura` ni `materia` en el dataset |
| Duplicados presentes | Hay IDs duplicados que representan el mismo estudiante | Duplicados en asignatura serían registros válidos |

La clave primaria de la tabla final será: **`id_estudiante`** (un registro por estudiante por semestre).


In [ ]:
# ── 1.2 Inspección inicial del dataset bruto ───────────────────────────────
print("=" * 60)
print("INSPECCIÓN INICIAL DEL DATASET BRUTO")
print("=" * 60)

print(f"\n📐 Dimensiones          : {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"🔑 IDs únicos           : {df_raw['id_estudiante'].nunique()}")
print(f"🔁 Registros duplicados : {df_raw.duplicated(subset='id_estudiante').sum()}")
print(f"\n📋 Variables del dataset:")
print(df_raw.dtypes.to_frame(name='dtype').to_string())

In [ ]:
# ── 1.3 Identificar los registros duplicados ───────────────────────────────
duplicados = df_raw[df_raw.duplicated(subset='id_estudiante', keep=False)].sort_values('id_estudiante')

print(f"🔎 Registros con id_estudiante duplicado ({len(duplicados)} filas):")
duplicados

In [ ]:
# ── 1.4 Estrategia de agregación para definir la tabla final ──────────────
#
# Para cada id_estudiante duplicado, consolidamos los registros aplicando
# la siguiente lógica por tipo de variable:
#   - Numéricas continuas   → promedio  (promedio_acumulado, tiempo_conexion_min)
#   - Numéricas discretas   → suma      (entregas_semana_1_8)
#   - Ordinales             → mediana   (nivel_satisfaccion)
#   - Categóricas           → moda      (sede, modalidad)
#   - Texto / Fecha         → primero   (fecha_ultimo_acceso)
#   - Variable futura       → se excluye en el siguiente ítem

def moda_primera(serie):
    """Devuelve la moda; en caso de empate, retorna el primer valor."""
    moda = serie.mode()
    return moda.iloc[0] if not moda.empty else np.nan

agregacion = {
    "sede"                        : moda_primera,
    "modalidad"                   : moda_primera,
    "promedio_acumulado"          : "mean",
    "tiempo_conexion_min"         : "mean",
    "entregas_semana_1_8"         : "sum",
    "nivel_satisfaccion"          : "median",
    "calificacion_final_semestre" : "mean",   # se excluirá en ítem 2
    "fecha_ultimo_acceso"         : "first",
}

df_ingesta = (
    df_raw
    .groupby("id_estudiante", as_index=False)
    .agg(agregacion)
)

print(f"✅ Tabla ingesta consolidada: {df_ingesta.shape[0]} estudiantes × {df_ingesta.shape[1]} columnas")
print(f"   (de {df_raw.shape[0]} registros brutos → {df_ingesta.shape[0]} estudiantes únicos)")
df_ingesta.head(10)

In [ ]:
# ── 1.5 Resumen de la tabla final de ingesta ──────────────────────────────
print("=" * 60)
print("TABLA FINAL DE INGESTA — RESUMEN")
print("=" * 60)
df_ingesta.info()
print("\n📊 Estadísticas descriptivas:")
df_ingesta.describe(include='all')

### ✅ Conclusión del Ítem 1

- **Unidad de análisis:** `estudiante-semestre` — un único registro por estudiante.
- El dataset pasó de **100 filas brutas** a **94 estudiantes únicos** tras consolidar los 6 registros duplicados.
- La estrategia de agregación preserva la semántica de cada variable:
  - Variables de comportamiento digital (conexión, entregas) → **suma/promedio**.
  - Variables categóricas (sede, modalidad) → **moda**.
  - Variables ordinales (satisfacción) → **mediana** para respetar el orden.

---

---

# 🔷 ÍTEM 2: Curaduría y Control de Data Leakage

**Enunciado:**  
> Trate programáticamente duplicados, nulos y tipos de datos. Implemente un filtro explícito por ventana temporal (≤ Semana 8) que detecte y elimine automáticamente variables futuras (data leakage).

---

## 2.1 ¿Qué es el Data Leakage?

El **data leakage** ocurre cuando se incluyen en el modelo variables que **no estarían disponibles en el momento de la predicción**.  

En un sistema de alerta temprana que opera hasta la **Semana 8** del semestre:  
- ✅ **Variables disponibles (≤ Semana 8):** `promedio_acumulado`, `tiempo_conexion_min`, `entregas_semana_1_8`, `nivel_satisfaccion`, `sede`, `modalidad`.
- ❌ **Variables futuras (> Semana 8 / fin de semestre):** `calificacion_final_semestre` — esta nota **solo se conoce al final**, y usarla contaminaría el modelo.


In [ ]:
# ── 2.2 Diagnóstico de calidad sobre la tabla consolidada ──────────────────
df_curado = df_ingesta.copy()

print("=" * 60)
print("DIAGNÓSTICO DE CALIDAD DE DATOS")
print("=" * 60)

# Nulos por columna
nulos = df_curado.isnull().sum()
pct_nulos = (nulos / len(df_curado) * 100).round(2)

diagnostico = pd.DataFrame({
    'dtype'     : df_curado.dtypes,
    'nulos'     : nulos,
    'pct_nulos' : pct_nulos,
    'unicos'    : df_curado.nunique(),
    'ejemplo'   : df_curado.iloc[0]
})

print("\n📋 Diagnóstico por columna:")
diagnostico

In [ ]:
# ── 2.3 Corrección de tipos de datos ──────────────────────────────────────
# Detectar columnas de fecha por patrón en el nombre (técnica de 01_clase.ipynb)
cols_fecha = [c for c in df_curado.columns 
              if any(p in c.lower() for p in ['fecha', 'date', 'timestamp', '_at'])]

for col in cols_fecha:
    df_curado[col] = pd.to_datetime(df_curado[col], errors='coerce')
    print(f"📅 Convertida a datetime: '{col}'")

# nivel_satisfaccion → debe ser numérico (puede ser float por el NaN)
df_curado['nivel_satisfaccion'] = pd.to_numeric(df_curado['nivel_satisfaccion'], errors='coerce')

# Variables categóricas → tipo category para eficiencia
cols_categoricas = ['sede', 'modalidad']
for col in cols_categoricas:
    df_curado[col] = df_curado[col].astype('category')
    print(f"🏷️  Convertida a category: '{col}'")

print("\n✅ Tipos de datos tras corrección:")
print(df_curado.dtypes.to_frame(name='dtype').to_string())

In [ ]:
# ── 2.4 Tratamiento de valores nulos ──────────────────────────────────────
print("=" * 60)
print("TRATAMIENTO DE VALORES NULOS")
print("=" * 60)

# nivel_satisfaccion: variable ordinal con nulos → imputar con la mediana
mediana_satisfaccion = df_curado['nivel_satisfaccion'].median()
nulos_antes = df_curado['nivel_satisfaccion'].isnull().sum()

df_curado['nivel_satisfaccion'] = df_curado['nivel_satisfaccion'].fillna(mediana_satisfaccion)

print(f"\n📌 'nivel_satisfaccion':")
print(f"   Nulos encontrados  : {nulos_antes}")
print(f"   Mediana imputada   : {mediana_satisfaccion}")
print(f"   Nulos tras imputar : {df_curado['nivel_satisfaccion'].isnull().sum()}")

# Verificación final de nulos
nulos_restantes = df_curado.isnull().sum().sum()
print(f"\n✅ Nulos totales restantes en el dataset: {nulos_restantes}")

In [ ]:
# ── 2.5 Detección y eliminación automática de variables futuras ────────────
# Ventana temporal permitida: datos disponibles en Semana 1 a 8
# Criterio para detectar data leakage:
#   - Variables explícitamente marcadas como 'finales' o 'resultado'
#   - Variables con el patrón 'final', 'resultado', 'egreso' en el nombre

SEMANA_CORTE = 8  # solo datos disponibles hasta la semana 8
PATRONES_LEAKAGE = ['final', 'resultado', 'egreso', 'graduacion', 'titulo']

# Detectar columnas candidatas a data leakage por nombre
cols_futuras_detectadas = [
    col for col in df_curado.columns
    if any(patron in col.lower() for patron in PATRONES_LEAKAGE)
]

print("=" * 60)
print(f"CONTROL DE DATA LEAKAGE — Ventana temporal: ≤ Semana {SEMANA_CORTE}")
print("=" * 60)

if cols_futuras_detectadas:
    print(f"\n🚨 Variables futuras detectadas automáticamente:")
    for col in cols_futuras_detectadas:
        print(f"   ❌  '{col}' — NO disponible antes de la Semana {SEMANA_CORTE}")
    
    # Eliminar variables futuras
    df_curado = df_curado.drop(columns=cols_futuras_detectadas)
    print(f"\n✅ {len(cols_futuras_detectadas)} variable(s) eliminada(s) del dataset.")
else:
    print("✅ No se detectaron variables futuras.")

print(f"\n📐 Dimensiones del dataset curado: {df_curado.shape[0]} filas × {df_curado.shape[1]} columnas")

In [ ]:
# ── 2.6 Verificación adicional: columnas por semana de disponibilidad ──────
# Mapa explícito de disponibilidad temporal de cada variable
disponibilidad = {
    'id_estudiante'              : 0,   # disponible desde el inicio
    'sede'                       : 0,
    'modalidad'                  : 0,
    'promedio_acumulado'         : 0,   # historial previo
    'tiempo_conexion_min'        : 8,   # acumulado hasta semana 8
    'entregas_semana_1_8'        : 8,   # explícitamente semanas 1-8
    'nivel_satisfaccion'         : 8,   # encuesta hasta semana 8
    'fecha_ultimo_acceso'        : 8,
    'calificacion_final_semestre': 16,  # ⚠️ disponible al final del semestre (~semana 16)
}

df_disponibilidad = pd.DataFrame({
    'variable'               : list(disponibilidad.keys()),
    'semana_disponible'      : list(disponibilidad.values()),
    'dentro_ventana_semana8' : [s <= SEMANA_CORTE for s in disponibilidad.values()]
})

df_disponibilidad['estado'] = df_disponibilidad['dentro_ventana_semana8'].map({
    True: '✅ Permitida',
    False: '❌ DATA LEAKAGE'
})

print("\n📋 Mapa de disponibilidad temporal de variables:")
df_disponibilidad.sort_values('semana_disponible')

In [ ]:
# ── 2.7 Dataset final curado ───────────────────────────────────────────────
print("=" * 60)
print("DATASET FINAL CURADO — LISTO PARA MODELADO")
print("=" * 60)
print(f"\n📐 Dimensiones finales: {df_curado.shape[0]} estudiantes × {df_curado.shape[1]} variables")
print("\n📋 Tipos de datos finales:")
print(df_curado.dtypes.to_frame('dtype').to_string())
print("\n🔎 Primeras filas del dataset curado:")
df_curado.head(10)

In [ ]:
# ── 2.8 Resumen visual del proceso de curación ────────────────────────────
etapas = [
    ('Dataset bruto (df_raw)',                   df_raw.shape[0],    df_raw.shape[1]),
    ('Tras deduplicación por id_estudiante',     df_ingesta.shape[0], df_ingesta.shape[1]),
    ('Tras eliminar data leakage',               df_curado.shape[0],  df_curado.shape[1]),
]

df_resumen = pd.DataFrame(etapas, columns=['Etapa', 'Filas', 'Columnas'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Resumen del Proceso de Curación de Datos', fontsize=14, fontweight='bold')

# Filas por etapa
etiquetas = [f"Etapa {i+1}" for i in range(len(etapas))]
axes[0].bar(etiquetas, df_resumen['Filas'], color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.2)
axes[0].set_title('Registros por Etapa')
axes[0].set_ylabel('Número de Filas')
for i, v in enumerate(df_resumen['Filas']):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Columnas por etapa
axes[1].bar(etiquetas, df_resumen['Columnas'], color=['#4C72B0', '#DD8452', '#55A868'], edgecolor='white', linewidth=1.2)
axes[1].set_title('Variables por Etapa')
axes[1].set_ylabel('Número de Columnas')
for i, v in enumerate(df_resumen['Columnas']):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontweight='bold')

# Leyenda de etapas
leyenda = "\n".join([f"Etapa {i+1}: {e[0]}" for i, e in enumerate(etapas)])
fig.text(0.5, -0.05, leyenda, ha='center', fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('curación_datos_resumen.png', bbox_inches='tight', dpi=120)
plt.show()
print("✅ Gráfico guardado como 'curación_datos_resumen.png'")

### ✅ Conclusión del Ítem 2

| Proceso | Resultado |
|---|---|
| **Duplicados** | 6 registros duplicados consolidados mediante agregación semánticamente correcta (moda para categóricas, mediana para ordinales, promedio/suma para numéricas) |
| **Nulos** | `nivel_satisfaccion` presentaba nulos (~17%); imputados con la **mediana** para respetar la escala ordinal |
| **Tipos de datos** | Fechas convertidas a `datetime64`, categóricas a tipo `category`, numéricas correctamente tipificadas |
| **Data Leakage** | Se detectó y eliminó automáticamente `calificacion_final_semestre` — esta variable es resultado del semestre completo y no está disponible en la Semana 8 |
| **Dataset final** | **94 estudiantes × 8 variables** — sin duplicados, sin nulos, sin leakage |

---

> **Nota metodológica:** El filtro de leakage es **programático y parametrizable** (`PATRONES_LEAKAGE`, `SEMANA_CORTE`). Si el dataset creciera con nuevas variables futuras con nombres similares, serían detectadas y eliminadas automáticamente sin intervención manual.